# Financial News Sentiment Prediction using Deep Learning (RNN, LSTM, GRU)

**Task:** classify finance tweets as **Bearish**, **Bullish** or **Neutral**
**Dataset:** `zeroshot/twitter-financial-news-sentiment` (Hugging Face)

**What I do in this notebook (step by step):**

1. Setup and imports
2. Load the data
3. Look at the data
4. Data quality checks
5. Clean the text
6. Exploratory data analysis (plots)
7. Build a vocabulary and convert text to numbers
8. PyTorch Dataset and DataLoader
9. Build the RNN / LSTM / GRU model
10. Training and evaluation functions
11. Train all three models
12. Compare the models
13. Save the best model (used by `app.py`)
14. Final comparison of the three models
15. Predict the sentiment of any paragraph
16. Try your own text and save the files
17. Conclusion

Every code cell has a short explanation above it saying what it does and why.

**Running this on Google Colab:**
1. Upload this notebook to Colab (or open it from Google Drive).
2. Optional but faster: *Runtime > Change runtime type > T4 GPU*.
3. Click *Runtime > Run all*.

Colab already has most libraries installed (pandas, torch, scikit-learn...). Only `datasets` is installed in the
first code cell.

## 1. Setup and imports

**What this cell does:** installs the one library that Google Colab does not have by default (`datasets`, used to
download the data from Hugging Face). If you run this notebook on your own computer, install everything first with
`pip install -r requirements.txt`.

**Why:** pandas/numpy for data, matplotlib/seaborn for plots, scikit-learn for metrics, torch for the deep learning
models, `datasets` for downloading the dataset.

In [ ]:
# Google Colab already has pandas, numpy, matplotlib, seaborn, scikit-learn and torch.
# Only the "datasets" library has to be installed.
# (On your own computer you can run:  pip install -r requirements.txt)
%pip install -q datasets

**What this cell does:** imports all the libraries and fixes the random seed.

**Why the seed:** neural networks start with random weights, so if I do not fix the seed I get slightly different
results every run. With a seed I can reproduce my numbers.
It also checks if a GPU is available; if not it uses the CPU.

In [ ]:
import os
import re
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Using device:", device)

# folders where I will save things
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

## 2. Load the data

**What this cell does:** downloads the dataset from Hugging Face. It already comes split into `train` and
`validation`, so I use these two splits as they are and do not make my own split (this also avoids data leakage).

**Note:** the dataset on Hugging Face has 9,543 train and 2,388 validation rows. The project brief says 9,938 and
2,486, so the numbers are a little different. I just use what the dataset gives.

In [ ]:
from datasets import load_dataset

raw_data = load_dataset("zeroshot/twitter-financial-news-sentiment")
print(raw_data)

**What this cell does:** converts the two splits into pandas DataFrames and saves them as CSV.

**Why:** pandas is easier to explore and clean, and the saved CSV files mean I do not need to download again.

In [ ]:
train_df = raw_data["train"].to_pandas()
val_df = raw_data["validation"].to_pandas()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

train_df.to_csv("data/train_raw.csv", index=False)
val_df.to_csv("data/validation_raw.csv", index=False)

## 3. Look at the data

**What this cell does:** creates the label dictionary (0 = Bearish, 1 = Bullish, 2 = Neutral) and adds a new
column `sentiment` with the label name, so the table is easy to read.

In [ ]:
label_map = {0: "Bearish", 1: "Bullish", 2: "Neutral"}

train_df["sentiment"] = train_df["label"].map(label_map)
val_df["sentiment"] = val_df["label"].map(label_map)

print(train_df.head(10))

**What this cell does:** counts how many tweets there are in each class.

**What I see:** the data is **imbalanced**: Neutral is much bigger than Bullish and Bearish. Because of this,
accuracy alone can be misleading, so I will also use **macro F1** (which treats every class equally).

In [ ]:
print("Train class counts:")
print(train_df["sentiment"].value_counts())
print()
print("Validation class counts:")
print(val_df["sentiment"].value_counts())

## 4. Data quality checks

**What this cell does:** checks for missing values, empty texts and duplicate tweets.

**Why:** bad data (empty or repeated rows) can hurt the model, so I check before training.

In [ ]:
print("Missing values in train:")
print(train_df.isnull().sum())
print()
print("Empty texts in train:", (train_df["text"].str.strip() == "").sum())
print("Empty texts in validation:", (val_df["text"].str.strip() == "").sum())
print("Duplicate texts in train:", train_df["text"].duplicated().sum())
print("Duplicate texts in validation:", val_df["text"].duplicated().sum())

**What this cell does:** checks if any tweet appears in both train and validation.

**Why this matters:** if the same tweet is in both, the model has already seen it in training and the validation
score would be fake (this is called **data leakage**). I want this number to be 0.

In [ ]:
common_texts = set(train_df["text"]).intersection(set(val_df["text"]))
print("Tweets present in both train and validation:", len(common_texts))

## 5. Clean the text

**What this cell does:** defines a function that cleans one tweet.

It removes: links (URLs), @mentions, HTML tags and the `#MarketScreener` hashtag, fixes some broken characters,
and removes extra spaces.

**What I keep:** the stock tickers like `$AAPL` and the finance words, because they carry the sentiment.

In [ ]:
def clean_text(text):
    text = str(text)

    # remove links
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    # remove @mentions
    text = re.sub(r"@\w+", " ", text)
    # remove html tags
    text = re.sub(r"<[^>]+>", " ", text)
    # remove the source hashtag that appears in this dataset
    text = re.sub(r"#MarketScreener\b", " ", text, flags=re.IGNORECASE)

    # fix some broken characters (encoding problems)
    bad_chars = {
        "\u00e2\u0080\u0099": "'",
        "\u00e2\u0080\u009c": '"',
        "\u00e2\u0080\u009d": '"',
        "\u00e2\u0080\u0093": "-",
        "\u00e2\u0080\u0094": "-",
        "\u00c2\u00a0": " ",
    }
    for bad, good in bad_chars.items():
        text = text.replace(bad, good)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

**What this cell does:** applies the cleaning function to every row and creates a new column `clean_text`.
It then prints 3 examples (before and after) to check that the cleaning works.

In [ ]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
val_df["clean_text"] = val_df["text"].apply(clean_text)

for i in range(3):
    print("BEFORE:", train_df.loc[i, "text"])
    print("AFTER :", train_df.loc[i, "clean_text"])
    print("-" * 80)

**What this cell does:** removes rows that became empty after cleaning (for example a tweet that was only a link),
then saves the cleaned files.

The saved `validation_clean.csv` is also used by the Streamlit app to draw the class distribution chart.

In [ ]:
train_before = len(train_df)
val_before = len(val_df)

train_df = train_df[train_df["clean_text"] != ""].reset_index(drop=True)
val_df = val_df[val_df["clean_text"] != ""].reset_index(drop=True)

print("Rows removed from train:", train_before - len(train_df))
print("Rows removed from validation:", val_before - len(val_df))
print("Final train shape:", train_df.shape)
print("Final validation shape:", val_df.shape)

train_df.to_csv("data/train_clean.csv", index=False)
val_df.to_csv("data/validation_clean.csv", index=False)

## 6. Exploratory Data Analysis (EDA)

**What this cell does:** counts the number of words and characters in each tweet and prints the summary
(mean, min, max...).

**Why:** I need to know how long the tweets are so I can choose a good maximum length for the model later.

In [ ]:
train_df["word_count"] = train_df["clean_text"].str.split().str.len()
train_df["char_count"] = train_df["clean_text"].str.len()

print(train_df[["word_count", "char_count"]].describe())

**What this cell does:** bar chart of the class distribution in the training set.

**What I see:** Neutral is the biggest class, so the data is imbalanced.

In [ ]:
plt.figure(figsize=(6, 4))
train_df["sentiment"].value_counts().plot(kind="bar", color=["#4472c4", "#3f9142", "#d64545"])
plt.title("Class distribution (train)")
plt.xlabel("Sentiment")
plt.ylabel("Number of tweets")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("data/class_distribution.png", dpi=120)
plt.show()

**What this cell does:** histogram of tweet lengths in words.

**What I see:** most tweets are short (around 8 to 15 words), so a small maximum length is enough.

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(train_df["word_count"], bins=30, color="#4472c4")
plt.title("Tweet length (words)")
plt.xlabel("Number of words")
plt.ylabel("Number of tweets")
plt.tight_layout()
plt.savefig("data/word_count_hist.png", dpi=120)
plt.show()

**What this cell does:** box plot of tweet length for each sentiment.

**Why:** to see if one sentiment has longer tweets than the others.

In [ ]:
train_df.boxplot(column="word_count", by="sentiment", figsize=(6, 4))
plt.title("Tweet length by sentiment")
plt.suptitle("")
plt.xlabel("Sentiment")
plt.ylabel("Word count")
plt.tight_layout()
plt.savefig("data/word_count_by_sentiment.png", dpi=120)
plt.show()

**What this cell does:** finds the most common words and the most common stock tickers.

**Why:** just to understand the vocabulary. Words like "to", "the", "of" are most common (stop words), and the
tickers show which companies are talked about most.

In [ ]:
def get_words(text):
    return re.findall(r"\$?[A-Za-z]+(?:[-'][A-Za-z]+)*", text.lower())

all_text = " ".join(train_df["clean_text"])
word_counter = Counter(get_words(all_text))

print("Total words:", sum(word_counter.values()))
print("Unique words:", len(word_counter))
print()
print("Top 15 words:")
for word, count in word_counter.most_common(15):
    print(word, count)

tickers = re.findall(r"\$[A-Za-z]+", all_text)
ticker_counter = Counter([t.upper() for t in tickers])
print()
print("Unique tickers:", len(ticker_counter))
print("Top 10 tickers:")
for ticker, count in ticker_counter.most_common(10):
    print(ticker, count)

## 7. Build a vocabulary and convert text to numbers

Neural networks cannot read words, only numbers. So I:

1. split each tweet into tokens (words),
2. give every word an ID number (the vocabulary),
3. turn each tweet into a list of IDs with the same length.

**Important:** I build the vocabulary from the **training data only**. If I used validation data too, information
from validation would leak into training.

In [ ]:
PAD = "<pad>"   # used to fill short tweets
UNK = "<unk>"   # used for words that are not in the vocabulary
MAX_LEN = 40    # every tweet will be exactly 40 tokens (the longest tweet is only ~31 words)

def tokenize(text):
    # keeps tickers like $AAPL as one token
    return re.findall(r"\$?[A-Za-z]+(?:[-'][A-Za-z]+)*|\d+", text.lower())

# count words in the TRAIN data only
counter = Counter()
for text in train_df["clean_text"]:
    counter.update(tokenize(text))

# keep only words that appear at least 2 times
vocab = {PAD: 0, UNK: 1}
for word, freq in counter.most_common(15000):
    if freq >= 2:
        vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))

**What this cell does:** a function that converts one text into a list of IDs. Unknown words get the `<unk>` ID,
long texts are cut at 40 tokens and short texts are filled with `<pad>` (0).

I test it on the first tweet to see the output.

In [ ]:
def text_to_ids(text, vocab, max_len=MAX_LEN):
    tokens = tokenize(text)
    ids = []
    for t in tokens:
        ids.append(vocab.get(t, vocab[UNK]))
    ids = ids[:max_len]                          # cut if too long
    ids = ids + [vocab[PAD]] * (max_len - len(ids))   # pad if too short
    return ids

print(train_df.loc[0, "clean_text"])
print(text_to_ids(train_df.loc[0, "clean_text"], vocab))

## 8. PyTorch Dataset and DataLoader

**What this cell does:** creates a `Dataset` class. PyTorch uses it to get one (tweet, label) pair at a time.

Then I create the `DataLoader`s, which give the data to the model in **batches of 64**.
The train loader shuffles the data every epoch; the validation loader does not.

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = list(texts)
        self.labels = list(labels)
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        ids = text_to_ids(self.texts[index], self.vocab)
        x = torch.tensor(ids, dtype=torch.long)
        y = torch.tensor(self.labels[index], dtype=torch.long)
        return x, y


BATCH_SIZE = 64

train_data = TweetDataset(train_df["clean_text"], train_df["label"], vocab)
val_data = TweetDataset(val_df["clean_text"], val_df["label"], vocab)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

# check one batch
xb, yb = next(iter(train_loader))
print("Input batch shape:", xb.shape)
print("Label batch shape:", yb.shape)

## 9. Build the RNN / LSTM / GRU model

**What this cell does:** one model class that can be a Simple RNN, an LSTM or a GRU (I choose with `rnn_type`).

**How the model works:**

1. **Embedding layer** turns every word ID into a vector of 100 numbers (the model learns these).
2. **Recurrent layer** (RNN / LSTM / GRU) reads the words one by one and keeps a memory in its hidden state.
3. I take the **last hidden state** as the summary of the whole tweet.
4. **Dropout** randomly switches off some neurons while training to reduce overfitting.
5. **Linear layer** turns the summary into 3 scores (Bearish, Bullish, Neutral).

LSTM and GRU have "gates" that help them remember long sequences better than a simple RNN.

In [ ]:
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=3, rnn_type="lstm", dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        if rnn_type == "rnn":
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif rnn_type == "lstm":
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        else:
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        output, hidden = self.rnn(x)

        # LSTM returns (h, c), the others return only h
        if isinstance(hidden, tuple):
            hidden = hidden[0]

        last = hidden[-1]          # hidden state of the last step
        last = self.dropout(last)
        return self.fc(last)

**What this cell does:** a quick test. I pass one batch through each of the three models to check that the output
shape is (64, 3): 64 tweets and 3 class scores each. If this fails, there is a bug in the model code.

In [ ]:
for t in ["rnn", "lstm", "gru"]:
    test_model = SentimentModel(len(vocab), rnn_type=t).to(device)
    out = test_model(xb.to(device))
    print(t, "output shape:", tuple(out.shape))

## 10. Training and evaluation functions

**What this cell does:** the `evaluate` function. It puts the model in evaluation mode (dropout off), runs it on
the validation data without updating weights, and returns the loss, accuracy and **macro F1**, plus the
predictions and true labels (for the confusion matrix later).

In [ ]:
def evaluate(model, loader):
    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * x.size(0)

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    result = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "preds": all_preds,
        "labels": all_labels,
    }
    return result

**What this cell does:** the training function.

For each epoch:
1. go through all training batches, compute the loss, do backpropagation and update the weights (Adam optimizer),
2. evaluate on the validation data,
3. keep a copy of the best weights (by validation macro F1).

**Early stopping:** if the validation macro F1 does not improve for 3 epochs in a row, I stop training. This
prevents overfitting (model memorizing the training data). At the end I load back the best weights.

**Gradient clipping:** I limit the size of the gradients. Simple RNNs often have exploding gradients, and this
helps keep the training stable.

In [ ]:
def train(model, train_loader, val_loader, epochs=15, lr=1e-3, patience=3, name="model"):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    best_f1 = -1
    best_weights = None
    no_improve = 0
    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        val = evaluate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val["loss"])
        history["val_acc"].append(val["accuracy"])
        history["val_f1"].append(val["macro_f1"])

        print(f"[{name}] epoch {epoch:2d} | train loss {train_loss:.4f} | val loss {val['loss']:.4f} | "
              f"val acc {val['accuracy']:.4f} | val macro F1 {val['macro_f1']:.4f}")

        # early stopping check
        if val["macro_f1"] > best_f1:
            best_f1 = val["macro_f1"]
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"[{name}] early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_weights)
    return model, history

## 11. Train all three models

**What this cell does:** trains the Simple RNN, the LSTM and the GRU one after another, with the **same settings**
so the comparison is fair (embedding 100, hidden size 128, learning rate 0.001, max 15 epochs).

For each model I store the trained model, the training history and the final validation results.
This can take a few minutes on a CPU.

In [ ]:
EMBED_DIM = 100
HIDDEN_DIM = 128
NUM_CLASSES = 3

models = {}
histories = {}
results = {}

for rnn_type in ["rnn", "lstm", "gru"]:
    print("=" * 80)
    print("Training", rnn_type.upper())
    print("=" * 80)

    model = SentimentModel(len(vocab), EMBED_DIM, HIDDEN_DIM, NUM_CLASSES, rnn_type)
    model, history = train(model, train_loader, val_loader, epochs=15, lr=1e-3, patience=3, name=rnn_type.upper())

    models[rnn_type] = model
    histories[rnn_type] = history
    results[rnn_type] = evaluate(model, val_loader)
    print()

## 12. Compare the models

**What this cell does:** puts the accuracy and macro F1 of the three models in one table, sorted from best to
worst. The best model is the one with the highest macro F1.

In [ ]:
compare_df = pd.DataFrame({
    "Model": [name.upper() for name in results],
    "Val Accuracy": [results[name]["accuracy"] for name in results],
    "Val Macro F1": [results[name]["macro_f1"] for name in results],
})
compare_df = compare_df.sort_values("Val Macro F1", ascending=False).reset_index(drop=True)
print(compare_df)

**What this cell does:** bar chart of the macro F1 of the three models, so I can see the difference quickly.

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(compare_df["Model"], compare_df["Val Macro F1"], color="#4472c4")
plt.title("Baseline models: validation macro F1")
plt.ylabel("Macro F1")
plt.ylim(0, 1)
for i, v in enumerate(compare_df["Val Macro F1"]):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.savefig("data/baseline_comparison.png", dpi=120)
plt.show()

**What this cell does:** training curves (training loss and validation loss per epoch) for each model.

**How to read it:** if the training loss keeps going down but the validation loss goes up, the model is
**overfitting**. This is the reason I use early stopping.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, histories):
    ax.plot(histories[name]["train_loss"], label="train loss")
    ax.plot(histories[name]["val_loss"], label="val loss")
    ax.set_title(name.upper() + " loss curve")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
plt.tight_layout()
plt.savefig("data/loss_curves.png", dpi=120)
plt.show()

**What this cell does:** prints the full classification report (precision, recall, F1 for each class) for every
model.

**How to read it:**
- *precision* = of the tweets the model called Bullish, how many were really Bullish
- *recall* = of the really Bullish tweets, how many did the model find
- *F1* = balance of both

In [ ]:
class_names = ["Bearish", "Bullish", "Neutral"]

for name in results:
    print("=" * 60)
    print(name.upper())
    print(classification_report(results[name]["labels"], results[name]["preds"],
                                target_names=class_names, zero_division=0))

**What this cell does:** confusion matrix for each model. Rows are the real labels and columns are the model's
predictions. Numbers on the diagonal are correct predictions; numbers outside the diagonal are mistakes.

**Error analysis idea:** look at which classes get confused most (usually Bearish/Bullish tweets predicted as
Neutral).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, results):
    cm = confusion_matrix(results[name]["labels"], results[name]["preds"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(name.upper() + " confusion matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("data/confusion_matrices.png", dpi=120)
plt.show()

**What this cell does:** shows some real mistakes from the best baseline model (tweets where the prediction was
wrong).

**Why:** numbers alone do not tell why the model fails. Reading the wrong examples helps me understand the errors.

In [ ]:
best_name = compare_df.iloc[0]["Model"].lower()
best_result = results[best_name]

error_df = val_df.copy()
error_df["predicted"] = [label_map[p] for p in best_result["preds"]]
error_df = error_df[error_df["sentiment"] != error_df["predicted"]]

print("Number of mistakes of", best_name.upper(), ":", len(error_df))
print()
for i, row in error_df.head(8).iterrows():
    print("TEXT     :", row["clean_text"])
    print("ACTUAL   :", row["sentiment"], "| PREDICTED:", row["predicted"])
    print()

## 13. Save the best baseline model

**What this cell does:** saves three things into the `models/` folder:
- the model weights (`best_baseline_model.pt`),
- the settings I used to create the model (`model_config.json`),
- the vocabulary (`vocab.json`).

`app.py` and `predict.py` load exactly these three files, so I do not need to retrain every time.

In [ ]:
print("Best baseline model:", best_name.upper(), "| macro F1:", round(compare_df.iloc[0]["Val Macro F1"], 4))

torch.save(models[best_name].state_dict(), "models/best_baseline_model.pt")

config = {
    "vocab_size": len(vocab),
    "embed_dim": EMBED_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_classes": NUM_CLASSES,
    "rnn_type": best_name,
    "max_len": MAX_LEN,
}
with open("models/model_config.json", "w") as f:
    json.dump(config, f, indent=2)

with open("models/vocab.json", "w") as f:
    json.dump(vocab, f)

print("Saved files in the models/ folder")

## 14. Final comparison of the three models

**What this cell does:** saves the comparison table to `data/model_comparison.csv` (the Streamlit app shows it)
and prints a small summary: which model is best, and how much better it is than the simple RNN.

This is the main result of the project.

In [ ]:
print(compare_df)
compare_df.to_csv("data/model_comparison.csv", index=False)

best_row = compare_df.iloc[0]
rnn_row = compare_df[compare_df["Model"] == "RNN"].iloc[0]

print()
print("Best model:", best_row["Model"], "with macro F1 =", round(best_row["Val Macro F1"], 4))
print("Simple RNN macro F1 =", round(rnn_row["Val Macro F1"], 4))
print("Improvement over the simple RNN:", round(best_row["Val Macro F1"] - rnn_row["Val Macro F1"], 4))

## 15. Predict the sentiment of any paragraph

The project asks for a script that takes a raw financial paragraph and gives its sentiment.

**What this cell does:** defines `predict_paragraph`.
1. split the paragraph into sentences,
2. clean each sentence and convert it to IDs,
3. get the probabilities from the model (softmax),
4. average the probabilities of all sentences and pick the class with the highest average.

The same logic is in `utils.py`, which `app.py` and `predict.py` use.

In [ ]:
def predict_paragraph(paragraph, model, vocab):
    model.eval()
    sentences = re.split(r"(?<=[.!?])\s+", paragraph.strip())
    sentences = [s for s in sentences if s.strip() != ""]

    all_probs = []
    for sentence in sentences:
        ids = text_to_ids(clean_text(sentence), vocab)
        x = torch.tensor([ids], dtype=torch.long).to(device)
        with torch.no_grad():
            probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
        all_probs.append(probs)

    avg_probs = np.mean(all_probs, axis=0)
    best_id = int(np.argmax(avg_probs))
    return label_map[best_id], {label_map[i]: float(avg_probs[i]) for i in range(3)}


text = "Shares of the company surge after record quarterly profit. Analysts raise their price targets."
label, probs = predict_paragraph(text, models[best_name], vocab)
print("Paragraph:", text)
print("Sentiment:", label)
print("Probabilities:", probs)

## 16. Try your own text and save the files

**What this cell does:** predicts the sentiment of a few of my own examples using the best model. I can change the
texts in the list and run the cell again (this is the cell to use when showing the project live).

In [ ]:
my_texts = [
    "$TSLA - Tesla upgraded to Buy at Goldman, price target raised.",
    "$AAPL - Apple downgraded at Morgan Stanley on weak iPhone demand.",
    "$MSFT to report quarterly earnings on Tuesday after the close.",
    "Shares surged after record quarterly profit. Analysts raised their targets. "
    "However the CFO warned about weaker demand next year.",
]

for t in my_texts:
    label, probs = predict_paragraph(t, models[best_name], vocab)
    print("TEXT     :", t)
    print("SENTIMENT:", label)
    print("PROBS    :", {k: round(v, 3) for k, v in probs.items()})
    print()

**What this cell does:** puts the `models/` and `data/` folders into one zip file, `saved_results.zip`, and starts
the download if I am on Google Colab.

**Why:** on Colab the files live on a temporary computer and disappear when the session ends. I download the zip to
keep the trained model and the charts, and to run `app.py` later on my own computer.

In [ ]:
import zipfile

with zipfile.ZipFile("saved_results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["models", "data"]:
        for root, dirs, files_in_folder in os.walk(folder):
            for file_name in files_in_folder:
                zf.write(os.path.join(root, file_name))

print("Created saved_results.zip")

try:
    from google.colab import files
    files.download("saved_results.zip")      # only works on Google Colab
except ImportError:
    print("Not on Colab, the zip file is in the project folder.")

## 17. Conclusion

**What I did**
- Loaded the official train / validation splits and checked missing values, duplicates and train/validation
  overlap (no leakage).
- Cleaned the tweets (links, mentions, HTML, source hashtag) and kept tickers and finance words.
- Built my own vocabulary from the train data only.
- Trained Simple RNN, LSTM and GRU with the same settings, early stopping and gradient clipping.
- Compared them using accuracy, macro F1, classification reports and confusion matrices.

**What to write in the report (fill in with the numbers from the tables above)**
- Best RNN-type model and its macro F1, and why it is better than the simple RNN
  (gates in LSTM / GRU handle longer sequences and reduce the vanishing gradient problem).
- How much better the best model is than the simple RNN (see the improvement printed in section 14).
- Which class is hardest (look at the confusion matrices; Bearish and Bullish tweets are often predicted as
  Neutral because Neutral has far more training examples).

**Possible improvements**
- Use class weights in the loss to handle the imbalance.
- Use pre-trained word vectors (GloVe) for the embedding layer.
- Try a bidirectional LSTM.

**Project files**
- `app.py` - Streamlit dashboard (`streamlit run app.py`)
- `predict.py` - command line script (`python predict.py "your paragraph"`)
- `utils.py` - shared functions used by both